# Lab 5, Day 1 — Exploration and Cleaning

Explore, profile, and clean the Titanic dataset, then split it and check your column
groups - no model yet. See `Lab5_Day1_Instructions.md` for the full walkthrough.

This notebook is just a shell: it gives you a place to write and document your work,
but the profiling, the decisions, and the reasoning are yours.

In [1]:
import pandas as pd
import numpy as np

from data import load_titanic

df, source = load_titanic()
print('source:', source)
df.head()


Loaded real Titanic from OpenML  (1309, 14)
source: openml


,Pclass,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## Step 1: Load and profile

In [2]:
# TODO: df.info(), df.describe(include='all').T, and missingness percentage per
# column, sorted descending
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   Pclass     1309 non-null   int64   
 1   Survived   1309 non-null   int64   
 2   Name       1309 non-null   str     
 3   Sex        1309 non-null   category
 4   Age        1046 non-null   float64 
 5   SibSp      1309 non-null   int64   
 6   Parch      1309 non-null   int64   
 7   Ticket     1309 non-null   str     
 8   Fare       1308 non-null   float64 
 9   Cabin      295 non-null    str     
 10  Embarked   1307 non-null   category
 11  boat       486 non-null    str     
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    str     
dtypes: category(2), float64(3), int64(4), str(5)
memory usage: 185.0 KB


In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Pclass,1309.0,NaN,NaN,NaN,2.294882,0.837836,1.0,2.0,3.0,3.0,3.0
Survived,1309.0,NaN,NaN,NaN,0.381971,0.486055,0.0,0.0,0.0,1.0,1.0
Name,1309,1307,"Connolly, Miss. Kate",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,1309,2,male,843,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,1046.0,NaN,NaN,NaN,29.881135,14.4135,0.1667,21.0,28.0,39.0,80.0
SibSp,1309.0,NaN,NaN,NaN,0.498854,1.041658,0.0,0.0,0.0,1.0,8.0
Parch,1309.0,NaN,NaN,NaN,0.385027,0.86556,0.0,0.0,0.0,0.0,9.0
Ticket,1309,929,CA. 2343,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,1308.0,NaN,NaN,NaN,33.295479,51.758668,0.0,7.8958,14.4542,31.275,512.3292
Cabin,295,186,C23 C25 C27,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
(df.isna().mean() * 100).sort_values(ascending=False)

body         90.756303
Cabin        77.463713
boat         62.872422
home.dest    43.086325
Age          20.091673
Embarked      0.152788
Fare          0.076394
Pclass        0.000000
Survived      0.000000
Name          0.000000
Sex           0.000000
SibSp         0.000000
Parch         0.000000
Ticket        0.000000
dtype: float64

## Step 2: The most skipped checks - does missingness predict the target?

In [5]:
# TODO: for at least Age and Cabin, compare Survived rates between rows where the
# column is missing vs not. Is there a difference worth caring about?
a = sum(df["Age"].isna())

c = sum(df["Cabin"].isna())

print(f"Age not documentated for {a} passengers")
print(f"Cabin Number not documented for {c} passengers")

Age not documentated for 263 passengers
Cabin Number not documented for 1014 passengers


In [6]:
df.groupby(df['Cabin'].isna())['Survived'].agg(
        count='count',
        survival_rate='mean'
    ).rename(index={True: 'Missing (NaN)', False: 'Recorded (Present)'})

,count,survival_rate
Cabin,,
Recorded (Present),295,0.654237
Missing (NaN),1014,0.302761


In [7]:
df.groupby(df['Age'].isna())['Survived'].agg(
        count='count',
        survival_rate='mean'
    ).rename(index={True: 'Missing (NaN)', False: 'Recorded (Present)'})

,count,survival_rate
Age,,
Recorded (Present),1046,0.408222
Missing (NaN),263,0.277567


Cabin has a lower fill rate but when present predicts survival much better than when age is filled (which has a much higher fill rate)

## Step 3: Look for impossible values and placeholders

In [8]:
# TODO: check unique values in object columns, zero/negative fares, absurd ages -
# anything that looks like a placeholder rather than real missingness
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}:")
    print(df[col].unique())


Name:
<ArrowStringArray>
[                  'Allen, Miss. Elisabeth Walton',
                  'Allison, Master. Hudson Trevor',
                    'Allison, Miss. Helen Loraine',
            'Allison, Mr. Hudson Joshua Creighton',
 'Allison, Mrs. Hudson J C (Bessie Waldo Daniels)',
                             'Anderson, Mr. Harry',
               'Andrews, Miss. Kornelia Theodosia',
                          'Andrews, Mr. Thomas Jr',
   'Appleton, Mrs. Edward Dale (Charlotte Lamson)',
                         'Artagaveytia, Mr. Ramon',
 ...
                             'Yasbeck, Mr. Antoni',
         'Yasbeck, Mrs. Antoni (Selini Alexander)',
                            'Youseff, Mr. Gerious',
                               'Yousif, Mr. Wazli',
                           'Yousseff, Mr. Gerious',
                            'Zabour, Miss. Hileni',
                           'Zabour, Miss. Thamine',
                       'Zakarian, Mr. Mapriededer',
                             'Zak

/var/folders/1j/nyl_qv9x4nscnrchs5b9jcpw0000gn/T/ipykernel_43331/333898383.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [9]:
bad_fares = df[df["Fare"] <= 0]
print(f"Rows with Fare <= 0: {len(bad_fares)}")
bad_fares[["Name", "Pclass", "Ticket", "Fare", "Embarked", "home.dest"]]

Rows with Fare <= 0: 17


,Name,Pclass,Ticket,Fare,Embarked,home.dest
7,"Andrews, Mr. Thomas Jr",1,112050,0.0,S,"Belfast, NI"
70,"Chisholm, Mr. Roderick Robert Crispin",1,112051,0.0,S,"Liverpool, England / Belfast"
125,"Fry, Mr. Richard",1,112058,0.0,S,NaN
150,"Harrison, Mr. William",1,112059,0.0,S,NaN
170,"Ismay, Mr. Joseph Bruce",1,112058,0.0,S,Liverpool
223,"Parr, Mr. William Henry Marsh",1,112052,0.0,S,Belfast
234,"Reuchlin, Jonkheer. John George",1,19972,0.0,S,"Rotterdam, Netherlands"
363,"Campbell, Mr. William",2,239853,0.0,S,Belfast
384,"Cunningham, Mr. Alfred Fleming",2,239853,0.0,S,Belfast
410,"Frost, Mr. Anthony Wood 'Archie'",2,239854,0.0,S,Belfast


In [10]:
df[df["Age"] <= 0]

,Pclass,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,boat,body,home.dest


In [11]:
print("Age range:", df["Age"].min(), "-", df["Age"].max())

Age range: 0.1667 - 80.0


No bad ages

In [12]:
#Placeholder check
for col in df.select_dtypes(include="object").columns:
    stripped = df[col].astype(str).str.strip().str.lower()
    weird = stripped[stripped.isin(["", "?", "na", "n/a", "none", "null", "-", "unknown"])]
    if len(weird):
        print(col, weird.value_counts())

/var/folders/1j/nyl_qv9x4nscnrchs5b9jcpw0000gn/T/ipykernel_43331/1344673008.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [13]:
#Leakage Check
candidate_cols = [c for c in df.columns if c != "Survived"]

for col in candidate_cols:
    present = df[col].notna()
    survival_if_present = df.loc[present, "Survived"].mean()
    survival_if_missing = df.loc[~present, "Survived"].mean()
    print(f"{col:12s} present_rate={present.mean():.2f}  "
          f"survival|present={survival_if_present:.2f}  "
          f"survival|missing={survival_if_missing:.2f}")

Pclass       present_rate=1.00  survival|present=0.38  survival|missing=nan
Name         present_rate=1.00  survival|present=0.38  survival|missing=nan
Sex          present_rate=1.00  survival|present=0.38  survival|missing=nan
Age          present_rate=0.80  survival|present=0.41  survival|missing=0.28
SibSp        present_rate=1.00  survival|present=0.38  survival|missing=nan
Parch        present_rate=1.00  survival|present=0.38  survival|missing=nan
Ticket       present_rate=1.00  survival|present=0.38  survival|missing=nan
Fare         present_rate=1.00  survival|present=0.38  survival|missing=0.00
Cabin        present_rate=0.23  survival|present=0.65  survival|missing=0.30
Embarked     present_rate=1.00  survival|present=0.38  survival|missing=1.00
boat         present_rate=0.37  survival|present=0.98  survival|missing=0.03
body         present_rate=0.09  survival|present=0.00  survival|missing=0.42
home.dest    present_rate=0.57  survival|present=0.47  survival|missing=0.27


## Step 4: Decide, and write it down

For each column with a problem, add a markdown cell (or a row in a table here) saying
what you did and why. Code with no commentary earns much less credit than the same
code with one sentence of justification.

**Cleaning Decisions**

**Age**

Problem found: 20% missing (263 of 1309 rows). Survival is 40.8% when present vs 27.8% when missing.

Decision: Keep the column, impute missing values with median Age grouped by Pclass/title, add an age_missing flag.

**Cabin**

Problem found: 77% missing, 186 unique values. Survival is 65.4% when present vs 30.3% when missing.

Decision: Drop the raw string, keep a binary has_cabin flag.

**Fare**

Problem found: 1 true NaN, plus 17 rows at exactly 0.0.

Decision: Treat the 17 zeros as valid (Pclass-1 passengers who weren't paying customers), impute only the 1 true NaN with the median Fare for that Pclass.

**Embarked**

Problem found: 2 missing values.

Decision: Impute with the mode, S.

**Name**

Problem found: 1307 unique values, unusable as a raw category.

Decision: Extract title (Mr/Mrs/Miss/Master/etc.), then drop the full name.

**Ticket**

Problem found: 929 unique alphanumeric values.

Decision: Drop, unless a ticket-prefix or group-size feature proves useful later.

**home.dest**

Problem found: 43% missing, 370 unique values, some entries look like two locations merged with no separator.

Decision: Drop.

**Placeholder strings**

Problem found: Checked all object columns for ?, unknown, na, n/a, none, empty string. None found.

Decision: No action needed; missingness is genuine NaN throughout.

**Target Leakage**

**boat**

Problem found: Present for 37% of rows; survival is 98% when present vs 3% when absent.

Decision: Drop before modeling.

**body**

Problem found: Present for 9% of rows; survival is 0% when present vs 42% when absent.

Decision: Drop before modeling.

## Step 5: Split first, before fitting anything (`pipeline.py`)

In [14]:
# TODO: build X (drop the target and any obvious identifiers/free text you won't
# use directly) and y, then train_test_split with stratify=y and a fixed random_state.
# Nothing should be .fit() on the full dataset before this split exists.
y = df["Survived"]

X = df.drop(columns=["Survived", "boat", "body"])

In [15]:
from pipeline import split_data, get_column_groups
X_train, X_test, y_train, y_test = split_data(X, y)

## Step 6: Column groups - and check them by eye

In [16]:
# TODO: split X's columns into numeric vs categorical. Print both lists and look at
# them - does anything dtype-based selection picked up actually belong in the other
# group? (Think about what Pclass really represents.)
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

X_train: (1047, 11)
X_test:  (262, 11)
y_train: (1047,)
y_test:  (262,)


In [17]:
print("Overall survival rate:", y.mean())
print("Training survival rate:", y_train.mean())
print("Testing survival rate:", y_test.mean())

Overall survival rate: 0.3819709702062643
Training survival rate: 0.38204393505253104
Testing survival rate: 0.3816793893129771


In [18]:
numeric_cols, categorical_cols = get_column_groups(X_train)

print("Numeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

all_grouped = set(numeric_cols) | set(categorical_cols)

Numeric columns:
['Age', 'SibSp', 'Parch', 'Fare']

Categorical columns:
['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked', 'home.dest', 'Pclass']


---
**Before you close this notebook today:**
- Save your split so Day 2 resumes rather than re-derives it - e.g.
  `joblib.dump({"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}, "split.joblib")`.
  Re-splitting tomorrow with a different `random_state` would silently invalidate every
  comparison you make against today's work.
- Confirm your split used `stratify=y` and a fixed `random_state`.
- Make sure Step 4's cleaning decisions are actually written down while the reasoning
  is fresh - reconstructing it tomorrow produces visibly thinner justifications.
- Keep this notebook and folder as-is. Day 2 is a new notebook here, not a restart.

In [19]:
import joblib

joblib.dump(
    {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    },
    "split.joblib"
)

['split.joblib']